In [ ]:

import os
import json
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from torchvision.transforms import v2
from torch.utils.data import Subset, DataLoader
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from medmnist import PathMNIST, INFO

# CONFIG
LR            = 0.001
EPOCHS        = 50
BATCH_SIZE    = 128
MILESTONES    = [20, 35, 45]
GAMMA         = 0.5
NUM_MC        = 100
PRUNE_FRAC    = 0.05
N_ITERATIONS  = 5
RESULTS_FILE  = "pruning_results.json"
CKPT_DIR      = "checkpoints"
FIG_DIR       = "figures"

BNN_PARAMS = {
    "prior_mu": 0.0,
    "prior_sigma": 1.0,
    "posterior_mu_init": 0.0,
    "posterior_rho_init": -3.0,
    "type": "Reparameterization",
    "moped_enable": False,
    "moped_delta": 0.5,
}

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device(
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available() else "cpu"
)
print(f"Device : {device}")

# Data

transform_aug = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomRotation(15),
    v2.ColorJitter(brightness=0.2),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

transform_eval = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

info        = INFO["pathmnist"]
class_names = list(info["label"].values())
N_CLASSES   = len(class_names)

# Trainset chargé deux fois : avec et sans aug (même indices → Subset cohérent)
trainset_aug  = PathMNIST(split="train", download=True, size=28, transform=transform_aug)
trainset_noaug = PathMNIST(split="train", download=True, size=28, transform=transform_eval)
valset        = PathMNIST(split="val",   download=True, size=28, transform=transform_eval)
testset       = PathMNIST(split="test",  download=True, size=28, transform=transform_eval)

valloader  = DataLoader(valset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

N_TRAIN = len(trainset_aug)
print(f"Trainset : {N_TRAIN} | Val : {len(valset)} | Test : {len(testset)}")
print(f"Classes  : {class_names}")

# MODEL BUILDERS
def build_bnn():
    net = torchvision.models.efficientnet_b0(progress=False)
    net.classifier[1] = nn.Linear(1280, N_CLASSES)
    dnn_to_bnn(net, BNN_PARAMS)
    return net.to(device)

def build_effnet():
    net = torchvision.models.efficientnet_b0(progress=False)
    net.classifier[1] = nn.Linear(1280, N_CLASSES)
    return net.to(device)

# TRAINING
def train_bnn(train_subset, epochs, tag=""):
    net = build_bnn()
    loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)
    n_train = len(train_subset)
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        net.train()
        run_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.squeeze(1).to(device)
            optimizer.zero_grad()
            out = net(inputs)
            kl = get_kl_loss(net)
            loss = nn.CrossEntropyLoss()(out, labels) + kl / n_train
            loss.backward()
            optimizer.step()
            run_loss += loss.item()
        train_losses.append(run_loss / len(loader))
        scheduler.step()

        net.eval()
        run_val = 0.0
        with torch.no_grad():
            for inputs, labels in valloader:
                inputs, labels = inputs.to(device), labels.squeeze(1).to(device)
                run_val += criterion(net(inputs), labels).item()
        val_losses.append(run_val / len(valloader))

        if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
            print(f"  [{tag}] epoch {epoch+1}/{epochs} — train {train_losses[-1]:.3f} — val {val_losses[-1]:.3f}")

    return net, train_losses, val_losses


def train_effnet(train_subset, epochs, tag=""):
    net = build_effnet()
    loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        net.train()
        run_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.squeeze(1).to(device)
            optimizer.zero_grad()
            loss = criterion(net(inputs), labels)
            loss.backward()
            optimizer.step()
            run_loss += loss.item()
        train_losses.append(run_loss / len(loader))
        scheduler.step()

        net.eval()
        run_val = 0.0
        with torch.no_grad():
            for inputs, labels in valloader:
                inputs, labels = inputs.to(device), labels.squeeze(1).to(device)
                run_val += criterion(net(inputs), labels).item()
        val_losses.append(run_val / len(valloader))

        if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
            print(f"  [{tag}] epoch {epoch+1}/{epochs} — train {train_losses[-1]:.3f} — val {val_losses[-1]:.3f}")

    return net, train_losses, val_losses


def compute_mc_outputs(net, dataset, is_bnn=True):
    loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
    net.eval()
    mc_runs = []
    n_mc = NUM_MC if is_bnn else 1
    with torch.no_grad():
        for _ in range(n_mc):
            batch_probs = []
            for inputs, _ in loader:
                inputs = inputs.to(device)
                logits = net(inputs)
                batch_probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            mc_runs.append(np.concatenate(batch_probs, axis=0))
    return np.stack(mc_runs, axis=0)  # (MC, N, C)


def entropy_scores(mc_outputs):
    mean_p  = mc_outputs.mean(axis=0)                                    
    H       = -np.sum(mean_p * np.log(mean_p + 1e-10), axis=-1)          
    exp_H   = -np.mean(np.sum(mc_outputs * np.log(mc_outputs + 1e-10), axis=-1), axis=0)  
    MI      = H - exp_H
    aleat   = H - MI
    return H, MI, aleat


def entropy_per_class(mc_outputs, labels):
    H, MI, aleat = entropy_scores(mc_outputs)
    h_c, mi_c, al_c = [], [], []
    for c in range(N_CLASSES):
        mask = labels == c
        h_c.append(float(H[mask].mean())    if mask.any() else 0.0)
        mi_c.append(float(MI[mask].mean())  if mask.any() else 0.0)
        al_c.append(float(aleat[mask].mean()) if mask.any() else 0.0)
    return {"H": h_c, "MI": mi_c, "H_MI": al_c}

# Complete evaluation function that returns predictions, labels, MC outputs, and accuracy.
def evaluate_loader(net, loader, is_bnn=True):
    net.eval()
    all_preds, all_labels, all_outputs = [], [], []
    n_mc = NUM_MC if is_bnn else 1
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.squeeze(1).cpu().numpy()
            mc = []
            for _ in range(n_mc):
                logits = net(inputs)
                mc.append(F.softmax(logits, dim=-1).cpu().numpy())
            mc_out = np.stack(mc, axis=0)  
            mean_p = mc_out.mean(axis=0)   
            preds  = mean_p.argmax(axis=-1)
            all_preds.append(preds)
            all_labels.append(labels)
            all_outputs.append(mc_out)

    preds   = np.concatenate(all_preds,  axis=0)
    labels  = np.concatenate(all_labels, axis=0)
    outputs = np.concatenate(all_outputs, axis=1)
    acc     = float((preds == labels).mean())
    return preds, labels, outputs, acc


def compute_auce(mc_outputs, labels, n_bins=20):
    H, _, _ = entropy_scores(mc_outputs)
    preds   = mc_outputs.mean(axis=0).argmax(axis=-1)
    correct = (preds == labels).astype(float)

    order      = np.argsort(H)
    H_sorted   = H[order]
    cor_sorted = correct[order]
    N = len(H_sorted)
    bin_size = N // n_bins
    errors = []
    thresholds = []
    for i in range(n_bins):
        thresh = H_sorted[min((i + 1) * bin_size - 1, N - 1)]
        mask = H <= thresh
        err = 1.0 - cor_sorted[:min((i + 1) * bin_size, N)].mean()
        errors.append(float(err))
        thresholds.append(float(thresh / (H_sorted.max() + 1e-10)))
    auce = float(np.trapezoid(errors, thresholds))
    return auce, thresholds, errors


def compute_ace(mc_outputs, labels, n_bins=20):
    mean_p = mc_outputs.mean(axis=0)
    conf   = mean_p.max(axis=-1)
    preds  = mean_p.argmax(axis=-1)
    correct = (preds == labels).astype(float)

    order     = np.argsort(conf)
    conf_s    = conf[order]
    correct_s = correct[order]
    N = len(conf_s)
    bin_size = max(N // n_bins, 1)
    accs, confs = [], []
    for i in range(n_bins):
        sl = slice(i * bin_size, min((i + 1) * bin_size, N))
        if correct_s[sl].size == 0:
            continue
        accs.append(float(correct_s[sl].mean()))
        confs.append(float(conf_s[sl].mean()))
    ace = float(np.mean(np.abs(np.array(accs) - np.array(confs))))
    return ace, confs, accs


def compute_uncertain_when_inaccurate(mc_outputs, labels, n_thresholds=50):
    H, _, _ = entropy_scores(mc_outputs)
    preds   = mc_outputs.mean(axis=0).argmax(axis=-1)
    inaccurate = (preds != labels)
    thresholds = np.linspace(H.min(), H.max(), n_thresholds)
    fracs, p_unc_inacc = [], []
    for t in thresholds:
        uncertain = H >= t
        frac = uncertain.mean()
        p = (uncertain & inaccurate).sum() / max(inaccurate.sum(), 1)
        fracs.append(float(frac))
        p_unc_inacc.append(float(p))
    return fracs, p_unc_inacc


def compute_conf_vs_acc(mc_outputs, labels, n_thresholds=50):
    mean_p  = mc_outputs.mean(axis=0)
    conf    = mean_p.max(axis=-1)
    correct = (mean_p.argmax(axis=-1) == labels).astype(float)
    thresholds = np.linspace(conf.min(), conf.max(), n_thresholds)
    threshs, frac_acc = [], []
    for t in thresholds:
        mask = conf >= t
        if mask.sum() == 0:
            continue
        threshs.append(float(t))
        frac_acc.append(float(correct[mask].mean()))
    return threshs, frac_acc


def accuracy_per_class(preds, labels):
    accs = []
    for c in range(N_CLASSES):
        mask = labels == c
        accs.append(float((preds[mask] == c).mean()) if mask.any() else 0.0)
    return accs

# PRUNING LOOP
MODEL_CONFIGS = {
    "bnn_no_aug": {
        "is_bnn": True,
        "train_fn": train_bnn,
        "trainset_for_train": trainset_noaug,  
        "trainset_for_score": trainset_noaug,
    },
    "bnn_aug": {
        "is_bnn": True,
        "train_fn": train_bnn,
        "trainset_for_train": trainset_aug,
        "trainset_for_score": trainset_noaug, 
    },
    "effnet_aug": {
        "is_bnn": False,
        "train_fn": train_effnet,
        "trainset_for_train": trainset_aug,
        "trainset_for_score": trainset_noaug,
    },
}

#Crash recovery
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    print(f"Data loaded from {RESULTS_FILE}")
else:
    all_results = {}


def save_results():
    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)


for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\n{'='*70}\n  MODÈLE : {model_name}\n{'='*70}")

    if model_name not in all_results:
        all_results[model_name] = {}

    current_indices = np.arange(N_TRAIN)
    scoring_net     = None

    for it in range(0, N_ITERATIONS + 1):
        iter_key = str(it)

        if iter_key in all_results[model_name]:
            print(f"  [iter {it}] already computed, loading checkpoint...")
            ckpt_path = os.path.join(CKPT_DIR, f"{model_name}_iter{it}.pth")
            ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
            current_indices = ckpt["kept_indices"]

            if cfg["is_bnn"]:
                scoring_net = build_bnn()
            else:
                scoring_net = build_effnet()
            scoring_net.load_state_dict(ckpt["model_state_dict"])
            scoring_net.to(device)
            continue

        print(f"\n  Itération {it} | trainset : {len(current_indices)} images")

        if it > 0:
            score_subset = Subset(cfg["trainset_for_score"], current_indices)
            mc_out = compute_mc_outputs(scoring_net, score_subset, is_bnn=cfg["is_bnn"])
            H_score, _, _ = entropy_scores(mc_out)

            n_remove = int(round(PRUNE_FRAC * len(current_indices)))
            order = np.argsort(H_score)
            keep_local    = order[:-n_remove] if n_remove > 0 else order
            removed_local = order[-n_remove:] if n_remove > 0 else np.array([], dtype=int)
            print(f"    Remove {n_remove} imgs | H removed={H_score[removed_local].mean():.4f} | kept={H_score[keep_local].mean():.4f}")
            current_indices = current_indices[keep_local]

        train_subset = Subset(cfg["trainset_for_train"], current_indices)
        tag = f"{model_name}_iter{it}"
        net, train_losses, val_losses = cfg["train_fn"](train_subset, EPOCHS, tag=tag)

        ckpt_path = os.path.join(CKPT_DIR, f"{model_name}_iter{it}.pth")
        torch.save({
            "model_state_dict": net.state_dict(),
            "train_losses": train_losses,
            "val_losses": val_losses,
            "kept_indices": current_indices,
        }, ckpt_path)

        val_preds, val_labels, val_mc, val_acc     = evaluate_loader(net, valloader,  cfg["is_bnn"])
        test_preds, test_labels, test_mc, test_acc = evaluate_loader(net, testloader, cfg["is_bnn"])

        H_val, MI_val, aleat_val = entropy_scores(val_mc)

        ent_by_class_val = entropy_per_class(val_mc, val_labels)

        auce_val, auce_thresh, auce_err = compute_auce(val_mc, val_labels)
        ace_val, ace_confs, ace_accs    = compute_ace(val_mc, val_labels)
        ui_fracs, ui_vals               = compute_uncertain_when_inaccurate(val_mc, val_labels)
        ca_threshs, ca_accs             = compute_conf_vs_acc(val_mc, val_labels)

        acc_per_class = accuracy_per_class(test_preds, test_labels)

        iter_data = {
            "n_train": int(len(current_indices)),
            "val_acc": val_acc,
            "test_acc": test_acc,
            "val_H_mean": float(H_val.mean()),
            "val_MI_mean": float(MI_val.mean()),
            "val_aleatoric_mean": float(aleat_val.mean()),
            "entropy_by_class_val": ent_by_class_val,
            "acc_per_class_test": acc_per_class,
            "calibration": {
                "auce": auce_val,
                "auce_thresholds": auce_thresh,
                "auce_errors": auce_err,
                "ace": ace_val,
                "ace_confidences": ace_confs,
                "ace_accuracies": ace_accs,
                "uncertain_when_inaccurate_fracs": ui_fracs,
                "uncertain_when_inaccurate_vals": ui_vals,
                "conf_vs_acc_thresholds": ca_threshs,
                "conf_vs_acc_accs": ca_accs,
            },
            "train_losses": train_losses,
            "val_losses": val_losses,
        }
        all_results[model_name][iter_key] = iter_data
        save_results()

        scoring_net = net
        print(f"    val_acc={val_acc:.3f} | test_acc={test_acc:.3f} | AUCE={auce_val:.3f} | ACE={ace_val:.3f}")

    print(f"\n  [{model_name}] terminé.")

print(f"\All results saved to {RESULTS_FILE}")

COLORS = plt.cm.tab10.colors
ITER_COLORS = [COLORS[i] for i in range(N_ITERATIONS + 1)]
HATCHES = ["", "//", "\\\\", "xx", "..", "oo"]


def plot_entropy_per_class(model_name, model_data, split="val"):
    iters = sorted(model_data.keys(), key=int)
    x = np.arange(N_CLASSES)
    n_iters = len(iters)
    bar_w = 0.8 / (n_iters * 2)

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, it in enumerate(iters):
        d = model_data[it]["entropy_by_class_val"]
        offset_H  = (2 * i    ) * bar_w - 0.4 + bar_w / 2
        offset_MI = (2 * i + 1) * bar_w - 0.4 + bar_w / 2
        color = ITER_COLORS[int(it)]
        ax.bar(x + offset_H,  d["H"],  bar_w, color=color,  label=f"iter{it} — H",  alpha=0.9)
        ax.bar(x + offset_MI, d["MI"], bar_w, color=color,  label=f"iter{it} — MI", hatch="//", alpha=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels([f"C{i}" for i in range(N_CLASSES)])
    ax.set_ylabel("Nats")
    ax.set_title(f"Entropie / Mutual Information par classe — {model_name} (val)")
    ax.legend(fontsize=7, ncol=4)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{model_name}_entropy_per_class.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")


def plot_calibration(model_name, model_data):
    iters = sorted(model_data.keys(), key=int)
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle(f"Calibration — {model_name}", fontsize=13)

    ax_auce, ax_ace, ax_ui, ax_ca = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

    for it in iters:
        cal = model_data[it]["calibration"]
        color = ITER_COLORS[int(it)]
        label = f"iter{it} (AUCE={cal['auce']:.1%})"
        ax_auce.plot(cal["auce_thresholds"], cal["auce_errors"], color=color, label=label)

        label_ace = f"iter{it} (ACE={cal['ace']:.1%})"
        ax_ace.plot(cal["ace_confidences"], cal["ace_accuracies"], color=color, label=label_ace)

        ax_ui.plot(cal["uncertain_when_inaccurate_fracs"],
                   cal["uncertain_when_inaccurate_vals"], color=color, label=f"iter{it}")

        ax_ca.plot(cal["conf_vs_acc_thresholds"],
                   cal["conf_vs_acc_accs"], color=color, label=f"iter{it}")

    # Diagonale parfaite ACE
    ax_ace.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Parfait")
    ax_auce.set_xlabel("Normalized predictive uncertainty")
    ax_auce.set_ylabel("Error rate")
    ax_auce.set_title("AUCE")
    ax_auce.legend(fontsize=7)
    ax_auce.grid(alpha=0.3)

    ax_ace.set_xlabel("Confidence")
    ax_ace.set_ylabel("Accuracy")
    ax_ace.set_title("ACE")
    ax_ace.legend(fontsize=7)
    ax_ace.grid(alpha=0.3)

    ax_ui.set_xlabel("Fraction retained (most uncertain)")
    ax_ui.set_ylabel("P(uncertain | inaccurate)")
    ax_ui.set_title("Uncertain when Inaccurate")
    ax_ui.legend(fontsize=7)
    ax_ui.grid(alpha=0.3)

    ax_ca.set_xlabel("Confidence threshold")
    ax_ca.set_ylabel("Fraction accurate predictions")
    ax_ca.set_title("Confidence vs Accuracy")
    ax_ca.legend(fontsize=7)
    ax_ca.grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{model_name}_calibration.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")


def plot_accuracy_per_class(model_name, model_data):
    iters = sorted(model_data.keys(), key=int)
    x = np.arange(N_CLASSES)
    n_iters = len(iters)
    bar_w = 0.8 / n_iters

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, it in enumerate(iters):
        accs = model_data[it]["acc_per_class_test"]
        offset = i * bar_w - 0.4 + bar_w / 2
        ax.bar(x + offset, [a * 100 for a in accs], bar_w,
               color=ITER_COLORS[int(it)], label=f"iter{it}", alpha=0.9)

    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"Accuracy par classe — {model_name} (test)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{model_name}_accuracy_per_class.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")


def plot_val_uncertainty_curve(model_name, model_data):
    iters = sorted(model_data.keys(), key=int)
    xs    = [int(it) for it in iters]
    H_vals   = [model_data[it]["val_H_mean"]         for it in iters]
    MI_vals  = [model_data[it]["val_MI_mean"]         for it in iters]
    al_vals  = [model_data[it]["val_aleatoric_mean"]  for it in iters]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(xs, H_vals,  marker="o", label="H — Predictive Entropy")
    ax.plot(xs, MI_vals, marker="o", label="MI — BALD (epistemic)")
    ax.plot(xs, al_vals, marker="o", label="H − MI (aleatoric)")
    ax.set_xlabel("Pruning iteration")
    ax.set_ylabel("Nats (valloader)")
    ax.set_title(f"Validation uncertainty across pruning iterations — {model_name}")
    ax.set_xticks(xs)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{model_name}_val_uncertainty.png")
    plt.savefig(path, dpi=130)
    plt.close()
    print(f"  Saved : {path}")


print("\nGenerating plots...")
for model_name, model_data in all_results.items():
    print(f"\n  {model_name}")
    plot_entropy_per_class(model_name, model_data)
    plot_calibration(model_name, model_data)
    plot_accuracy_per_class(model_name, model_data)
    plot_val_uncertainty_curve(model_name, model_data)

print("\nDone.")

Device : cuda
Trainset : 89996 | Val : 10004 | Test : 7180
Classes  : ['adipose', 'background', 'debris', 'lymphocytes', 'mucus', 'smooth muscle', 'normal colon mucosa', 'cancer-associated stroma', 'colorectal adenocarcinoma epithelium']
Résultats existants chargés depuis pruning_results.json

  MODÈLE : bnn_no_aug
  [iter 0] déjà calculé, chargement du checkpoint...
  [iter 1] déjà calculé, chargement du checkpoint...
  [iter 2] déjà calculé, chargement du checkpoint...
  [iter 3] déjà calculé, chargement du checkpoint...
  [iter 4] déjà calculé, chargement du checkpoint...
  [iter 5] déjà calculé, chargement du checkpoint...

  [bnn_no_aug] terminé.

  MODÈLE : bnn_aug
  [iter 0] déjà calculé, chargement du checkpoint...
  [iter 1] déjà calculé, chargement du checkpoint...

  --- Itération 2 | trainset : 85496 images ---
    Retrait 4275 imgs | H retiré=1.1142 | gardé=0.3233
  [bnn_aug_iter2] epoch 10/50 — train 1.254 — val 1.859
  [bnn_aug_iter2] epoch 20/50 — train 0.434 — val 0.76